In [1]:
import os
import pandas as pd
import numpy as np
from datasets import load_dataset, Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
)
from sklearn.metrics import (
    precision_recall_fscore_support,
    precision_score,
    recall_score,
    f1_score,
    accuracy_score,
)
from sklearn.model_selection import StratifiedKFold

c:\Users\c24082331\OneDrive - Cardiff University\Desktop\RA(UniversalCEFR)\development\universalcefr\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
df_bt = pd.read_csv("files/welsh_back_translation_high_similarity.csv",
                    usecols=["back_translated_welsh", "cefr_level"])\
                .rename(columns={"back_translated_welsh": "text"})

In [3]:
df_bt

,text,cefr_level
0,Roedd Ma/ penbwrdd: fy nhad yn awr yn mynd i'r...,A1
1,Ar y acw: Byddwch yn troi at y dde yma. Ewch o...,A1
2,Ac: mae prynhawn da. Ar y tywydd yn ofnadwy! Y...,A1
3,Ble ydych chi'n byw?,A1
4,Beth ydych chi'n hoffi?,A1
...,...,...
365,Dylen nhw aros yn aros.,A2
366,Hoffwn i fynd i Affrica.,A2
367,A hoffech chi fynd i America?,A2
368,A fydden nhw'n mynd i'r Almaen?,A2


In [4]:
# Add missing columns
df_bt["title"] = "Back-translated A1/A2 sample"
df_bt["lang"] = "cy"
df_bt["source_name"] = "back_translation_pipeline"
df_bt["format"] = "text"
df_bt["category"] = "general"
df_bt["license"] = "CC-BY-SA" 

In [5]:
df_bt

,text,cefr_level,title,lang,source_name,format,category,license
0,Roedd Ma/ penbwrdd: fy nhad yn awr yn mynd i'r...,A1,Back-translated A1/A2 sample,cy,back_translation_pipeline,text,general,CC-BY-SA
1,Ar y acw: Byddwch yn troi at y dde yma. Ewch o...,A1,Back-translated A1/A2 sample,cy,back_translation_pipeline,text,general,CC-BY-SA
2,Ac: mae prynhawn da. Ar y tywydd yn ofnadwy! Y...,A1,Back-translated A1/A2 sample,cy,back_translation_pipeline,text,general,CC-BY-SA
3,Ble ydych chi'n byw?,A1,Back-translated A1/A2 sample,cy,back_translation_pipeline,text,general,CC-BY-SA
4,Beth ydych chi'n hoffi?,A1,Back-translated A1/A2 sample,cy,back_translation_pipeline,text,general,CC-BY-SA
...,...,...,...,...,...,...,...,...
365,Dylen nhw aros yn aros.,A2,Back-translated A1/A2 sample,cy,back_translation_pipeline,text,general,CC-BY-SA
366,Hoffwn i fynd i Affrica.,A2,Back-translated A1/A2 sample,cy,back_translation_pipeline,text,general,CC-BY-SA
367,A hoffech chi fynd i America?,A2,Back-translated A1/A2 sample,cy,back_translation_pipeline,text,general,CC-BY-SA
368,A fydden nhw'n mynd i'r Almaen?,A2,Back-translated A1/A2 sample,cy,back_translation_pipeline,text,general,CC-BY-SA


In [6]:
# Count how many samples are labeled A1 and A2
df_bt["cefr_level"].value_counts()

cefr_level
A1    243
A2    127
Name: count, dtype: int64

In [8]:
hf_dataset=Dataset.from_pandas(df_bt.reset_index(drop=True))

In [9]:
hf_dataset

Dataset({
    features: ['text', 'cefr_level', 'title', 'lang', 'source_name', 'format', 'category', 'license'],
    num_rows: 370
})

In [10]:
CEFR_LEVELS = ["A1", "A2", "B1", "B2", "C1", "C2"]
label2id = {lvl: i for i,lvl in enumerate(CEFR_LEVELS)}
labels = np.array([label2id[l] for l in hf_dataset["cefr_level"]])

In [11]:
model_name = "./eurobert_cefr_welsh_b2/best_model"
tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True, trust_remote_code=True)
data_collator = DataCollatorWithPadding(tokenizer)

In [12]:
def preprocess(batch):
    toks = tokenizer(batch["text"], truncation=True, max_length=256)
    toks["labels"] = [label2id[l] for l in batch["cefr_level"]]
    return toks

In [13]:
# Metrics
def compute_metrics(pred):
    logits, labels = pred
    preds = np.argmax(logits, axis=-1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, labels=list(range(len(CEFR_LEVELS))), zero_division=0
    )

    metrics = {}
    for i, label in enumerate(CEFR_LEVELS):
        metrics[f"{label}_precision"] = precision[i]
        metrics[f"{label}_recall"] = recall[i]
        metrics[f"{label}_f1"] = f1[i]

    metrics["eval_accuracy"] = accuracy_score(labels, preds)
    metrics["eval_weighted_f1"] = f1_score(labels, preds, average="weighted")
    metrics["eval_weighted_precision"] = precision_score(labels, preds, average="weighted")
    metrics["eval_weighted_recall"] = recall_score(labels, preds, average="weighted")
    return metrics

In [14]:
# Cross-validation setup
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
all_results = []

In [15]:
best_f1 = 0.0
best_trainer = None
best_tokenizer = None

for fold, (train_idx, val_idx) in enumerate(skf.split(hf_dataset, labels), start=1):
    print(f"\n Running Fold {fold}...")

    ds_train = hf_dataset.select(train_idx)
    ds_val = hf_dataset.select(val_idx)

    tok_train = ds_train.map(preprocess, batched=True, remove_columns=ds_train.column_names)
    tok_val = ds_val.map(preprocess, batched=True, remove_columns=ds_val.column_names)

    model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=len(CEFR_LEVELS),trust_remote_code=True)

    args = TrainingArguments(
        output_dir=f"./eurobert_cefr_welsh_DA_FTmodel/fold_{fold}",  
        num_train_epochs=3, 
        per_device_train_batch_size=2,              
        per_device_eval_batch_size=3,                
        eval_strategy="epoch",
        logging_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="eval_weighted_f1",
        greater_is_better=True,
        seed=42,
        learning_rate=3.6e-5,
        warmup_ratio=0.1,
        gradient_accumulation_steps=16,      
        optim="adamw_torch_fused",                   
        lr_scheduler_type="linear",                  
        adam_beta1=0.9,
        adam_beta2=0.999,
        adam_epsilon=1e-8,
        save_total_limit=1,
    )

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=tok_train,
        eval_dataset=tok_val,
        tokenizer=tokenizer,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
    )

    trainer.train()
    metrics = trainer.evaluate()

    # Track best trainer
    if metrics["eval_weighted_f1"] > best_f1:
        best_f1 = metrics["eval_weighted_f1"]
        best_trainer = trainer
        best_tokenizer = tokenizer

    # Store fold metrics
    row = {
        "Fold": fold,
        "All CEFR Levels Precision": metrics.get("eval_weighted_precision", 0.0),
        "All CEFR Levels Recall": metrics.get("eval_weighted_recall", 0.0),
        "All CEFR Levels F1": metrics.get("eval_weighted_f1", 0.0),
    }
    for level in ["A1", "A2", "B1", "B2", "C1", "C2"]:
        row[f"{level} Precision"] = metrics.get(f"eval_{level}_precision", 0.0)
        row[f"{level} Recall"] = metrics.get(f"eval_{level}_recall", 0.0)
        row[f"{level} F1"] = metrics.get(f"eval_{level}_f1", 0.0)

    all_results.append(row)


 Running Fold 1...


Map: 100%|██████████| 74/74 [00:00<00:00, 6120.17 examples/s]
C:\Users\c24082331\AppData\Local\Temp\ipykernel_50816\3231141536.py:39: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,Accuracy,Weighted F1,Weighted Precision,Weighted Recall,A1 Precision,A1 Recall,A1 F1,A2 Precision,A2 Recall,A2 F1,B1 Precision,B1 Recall,B1 F1,B2 Precision,B2 Recall,B2 F1,C1 Precision,C1 Recall,C1 F1,C2 Precision,C2 Recall,C2 F1
1,0.663900,0.634525,0.702703,0.710445,0.768399,0.702703,0.885714,0.632653,0.738095,0.538462,0.840000,0.656250,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2,0.335100,0.694619,0.716216,0.707338,0.705455,0.716216,0.759259,0.836735,0.796117,0.600000,0.480000,0.533333,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
3,0.172500,0.736327,0.743243,0.749844,0.778120,0.743243,0.875000,0.714286,0.786517,0.588235,0.800000,0.677966,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000



 Running Fold 2...


Map: 100%|██████████| 74/74 [00:00<00:00, 3763.35 examples/s]
C:\Users\c24082331\AppData\Local\Temp\ipykernel_50816\3231141536.py:39: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


...

Map: 100%|██████████| 74/74 [00:00<00:00, 4187.06 examples/s]
C:\Users\c24082331\AppData\Local\Temp\ipykernel_50816\3231141536.py:39: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,Accuracy,Weighted F1,Weighted Precision,Weighted Recall,A1 Precision,A1 Recall,A1 F1,A2 Precision,A2 Recall,A2 F1,B1 Precision,B1 Recall,B1 F1,B2 Precision,B2 Recall,B2 F1,C1 Precision,C1 Recall,C1 F1,C2 Precision,C2 Recall,C2 F1
1,0.636200,0.495627,0.702703,0.621408,0.796139,0.702703,0.685714,1.000000,0.813559,1.000000,0.153846,0.266667,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2,0.294600,0.374952,0.824324,0.821608,0.821870,0.824324,0.843137,0.895833,0.868687,0.782609,0.692308,0.734694,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
3,0.130300,0.344259,0.851351,0.854324,0.872536,0.851351,0.951220,0.812500,0.876404,0.727273,0.923077,0.813559,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000



 Running Fold 5...


Map: 100%|██████████| 74/74 [00:00<00:00, 9249.29 examples/s]
C:\Users\c24082331\AppData\Local\Temp\ipykernel_50816\3231141536.py:39: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,Accuracy,Weighted F1,Weighted Precision,Weighted Recall,A1 Precision,A1 Recall,A1 F1,A2 Precision,A2 Recall,A2 F1,B1 Precision,B1 Recall,B1 F1,B2 Precision,B2 Recall,B2 F1,C1 Precision,C1 Recall,C1 F1,C2 Precision,C2 Recall,C2 F1
1,0.560200,0.564497,0.743243,0.715762,0.748782,0.743243,0.737705,0.937500,0.825688,0.769231,0.384615,0.512821,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2,0.422000,0.758046,0.729730,0.715555,0.720721,0.729730,0.750000,0.875000,0.807692,0.666667,0.461538,0.545455,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
3,0.234700,0.494917,0.797297,0.798143,0.799289,0.797297,0.851064,0.833333,0.842105,0.703704,0.730769,0.716981,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000


In [16]:
# Save best-performing model from all folds
final_path = "./eurobert_cefr_welsh_DA_FTmodel/best_model"
best_trainer.save_model(final_path)
best_tokenizer.save_pretrained(final_path)
best_trainer.state.save_to_json(os.path.join(final_path, "trainer_state.json"))

In [17]:
# Convert to DataFrame
df = pd.DataFrame(all_results)

# Compute average row
average_row = df.drop(columns=["Fold"]).mean(numeric_only=True)
average_row["Fold"] = "Average"
df = pd.concat([df, pd.DataFrame([average_row])], ignore_index=True)

# Restructure columns
columns = [("Fold", "")] + [
    ("All CEFR Levels", "Precision"), ("All CEFR Levels", "Recall"), ("All CEFR Levels", "F1"),
    ("A1", "Precision"), ("A1", "Recall"), ("A1", "F1"),
    ("A2", "Precision"), ("A2", "Recall"), ("A2", "F1"),
    ("B1", "Precision"), ("B1", "Recall"), ("B1", "F1"),
    ("B2", "Precision"), ("B2", "Recall"), ("B2", "F1"),
    ("C1", "Precision"), ("C1", "Recall"), ("C1", "F1"),
    ("C2", "Precision"), ("C2", "Recall"), ("C2", "F1"),
]


df = df[[col[0] if col[1] == "" else f"{col[0]} {col[1]}" for col in columns]]
df.columns = pd.MultiIndex.from_tuples(columns)


In [18]:
df

Fold All CEFR Levels                            A1                      \
                 Precision    Recall        F1 Precision    Recall        F1   
0        1        0.778120  0.743243  0.749844  0.875000  0.714286  0.786517   
1        2        0.843436  0.824324  0.808414  0.800000  0.979592  0.880734   
2        3        0.822838  0.824324  0.823419  0.860000  0.877551  0.868687   
3        4        0.872536  0.851351  0.854324  0.951220  0.812500  0.876404   
4        5        0.799289  0.797297  0.798143  0.851064  0.833333  0.842105   
5  Average        0.823244  0.808108  0.806829  0.867457  0.843452  0.850889   

         A2                      ...   B1        B2                    C1  \
  Precision    Recall        F1  ...   F1 Precision Recall   F1 Precision   
0  0.588235  0.800000  0.677966  ...  0.0       0.0    0.0  0.0       0.0   
1  0.928571  0.520000  0.666667  ...  0.0       0.0    0.0  0.0       0.0   
2  0.750000  0.720000  0.734694  ...  0.0       0.0    0.0  0.0       0.0   
3  0.727273  0.923077  0.813559  ...  0.0       0.0    0.0  0.0       0.0   
4  0.703704  0.730769  0.716981  ...  0.0       0.0    0.0  0.0       0.0   
5  0.739557  0.738769  0.721973  ...  0.0       0.0    0.0  0.0       0.0   

                     C2              
  Recall   F1 Precision Recall   F1  
0    0.0  0.0       0.0    0.0  0.0  
1    0.0  0.0       0.0    0.0  0.0  
2    0.0  0.0       0.0    0.0  0.0  
3    0.0  0.0       0.0    0.0  0.0  
4    0.0  0.0       0.0    0.0  0.0  
5    0.0  0.0       0.0    0.0  0.0  

[6 rows x 22 columns]